<div style="border-left:5px solid #67CBEE;padding:10px 16px;background:#F7F9FB;color:#19222D;margin:12px 0"><b>AMD Radeon · YOLO26 · Day 01</b><br><span style="color:#536273">算法 → 数据 → 训练 → 评测 → 证据。每一个结论都应由实际运行结果支撑。</span><br><span style="color:#536273">Algorithm → Data → Training → Evaluation → Evidence. Every conclusion must be backed by real execution results.</span></div>

# 04 / Lab D · mAP、混淆与失败案例

**建议时长：35–45 分钟**　｜　学习路径：官方指标 → 固定阈值配对 → 错误归因

> 请按顺序运行；运行结果必须来自自己的 GPU 环境，不可把示例文字当作实测数据。

---

# 04 / Lab D · mAP, Confusion & Failure Cases

**Suggested time: 35–45 minutes** | Learning path: official metrics → fixed-threshold matching → error attribution

> Run the cells in order; every result must come from your own GPU environment — never present the sample text as measured data.

> ### 运行前 30 秒：确认 Kernel 选对了
> 本课程全部 Notebook 必须运行在 **`YOLO26 - AMD ROCm 7.14.1`** 内核上（看页面右上角的内核名）。
>
> - **怎么选**：菜单 **Kernel → Change Kernel → YOLO26 - AMD ROCm 7.14.1**（JupyterLab 点右上角内核名）。
> - **怎么确认**：运行第一个代码格后，`torch_hip` 是版本号而非 `None`，并打印出你的 AMD 显卡名。
> - **选错会怎样**：00 会被 ROCm 门禁当场拦截；若误用平台默认 `python3`，可能到 Lab A 才报出难懂的 `NotImplementedError`。
> - **列表里没有这个内核？** 在项目目录执行一次注册：
>   `.venv/bin/python -m ipykernel install --user --name yolo26-rocm7141 --display-name "YOLO26 - AMD ROCm 7.14.1"`
>   （Windows 见 `docs/ROCM_WINDOWS_SETUP.md`；Linux 见 `docs/RADEON_CLOUD_LINUX_SETUP.md` §2B）

---

> ### 30 seconds before you run: make sure the Kernel is right
> Every notebook in this course must run on the **`YOLO26 - AMD ROCm 7.14.1`** kernel (see the kernel name at the top right).
>
> - **How to select**: menu **Kernel → Change Kernel → YOLO26 - AMD ROCm 7.14.1** (JupyterLab: click the kernel name at the top right).
> - **How to confirm**: after the first code cell, `torch_hip` is a version string (not `None`) and your AMD GPU name is printed.
> - **What if you pick the wrong one**: Lab 00's ROCm gate stops you immediately; with the platform default `python3` you may instead hit an opaque `NotImplementedError` deep inside Lab A.
> - **Kernel not in the list?** Register it once from the project directory:
>   `.venv/bin/python -m ipykernel install --user --name yolo26-rocm7141 --display-name "YOLO26 - AMD ROCm 7.14.1"`
>   (Windows: `docs/ROCM_WINDOWS_SETUP.md`; Linux: `docs/RADEON_CLOUD_LINUX_SETUP.md` §2B)

### 学习地图（30 秒看完这张表再开始）

| 问题 | 答案 |
|---|---|
| 为什么学这个？ | 没有评价的训练是盲飞；错误归因决定下一步优化方向 |
| 需要先懂什么？ | 完成 03（要它的 best.pt）；Lab A 的 IoU 概念 |
| 我修改哪个参数？ | `MY_ERROR_INDEX` 与 `MY_ANALYSIS` 四字段（选一个错误深挖） |
| 成功长什么样？ | 官方 mAP 打印、PR/混淆矩阵图、左右对照失败图 |
| 失败怎么自救？ | 本页常见错误表；失败案例不足 5 张时如实报告 |
| 保存什么证据？ | 官方指标 JSON、固定阈值 JSON、errors.csv、归因 JSON |
| 求职价值？ | "讲一个你定位过的 badcase"——算法岗必问 |

---

### Lab Map (read this 30-second table first)

| Question | Answer |
|---|---|
| Why learn this? | Training without evaluation is flying blind; error attribution decides the next move |
| Prerequisites? | Lab 03 (its best.pt); the IoU idea from Lab A |
| What do I change? | `MY_ERROR_INDEX` and the four fields of `MY_ANALYSIS` (dig into one error) |
| What does success look like? | Official mAP printed, PR/confusion plots, side-by-side failure boards |
| If it fails? | The common-errors table; report honestly when fewer than 5 failure cases exist |
| Evidence to save? | Official-metrics JSON, fixed-threshold JSON, errors.csv, your attribution JSON |
| Job value? | "Walk me through a bad case you diagnosed" — asked in every algorithm interview |

## 实验目标
- 用 Ultralytics 官方验证器计算 Precision、Recall、mAP50、mAP50-95。
- 独立完成一次**固定 `conf=0.25`、IoU=0.50 的教学级 TP/FP/FN 匹配**（并非 COCO mAP 实现）。
- 查看 PR/混淆矩阵并生成至少五份真实失败案例。
- 不用 Test 集选择模型或调阈值。

**依赖**：先完整运行 03（需要它的 `best.pt` 与数据集登记）。本实验的预测都在 GPU 上完成，但单张推理很快。

`metrics.box.map` 才是官方 mAP50-95；我们的 TP/FP/FN 固定阈值分析只用于学习错误原因。

---

## Goal
- Compute Precision, Recall, mAP50 and mAP50-95 with the official Ultralytics validator.
- Independently run a **fixed-threshold `conf=0.25`, IoU=0.50 teaching-level TP/FP/FN matching** (this is NOT the COCO mAP algorithm).
- Inspect PR/confusion matrices and produce at least five genuine failure cases.
- Never use the test split to pick models or tune thresholds.

**Prerequisite**: fully run 03 first (its `best.pt` and dataset registration are required). Predictions run on the GPU but are fast per image.

`metrics.box.map` is the official mAP50-95; our fixed-threshold TP/FP/FN analysis exists only to learn about error causes.

### 常见错误与下一步
| 症状 | 原因 | 下一步 |
|---|---|---|
| `先完成 Lab C` 断言失败 | course_config 里没有 baseline_weights | 先完整运行 03（训练 + 保存 best.pt） |
| `lab_d_errors.csv` 是空的 | 该阈值下确实没有错误（或阈值过高） | 如实报告"0 个错误"；可选做困难样本分析，不要编造 |
| 官方 P/R 与固定阈值 P/R 差很多 | 两套口径本就不同 | 重读上面的双口径说明；报告时分开写、分开引用 |
| PR/混淆矩阵图不显示 | plots 输出目录因版本而异 | 查看 `val_metrics.save_dir` 的打印值并打开该目录 |

---

### Common Errors & Next Steps
| Symptom | Cause | Next step |
|---|---|---|
| `先完成 Lab C` assertion fires | no baseline_weights in course_config | fully run Lab 03 (training + saved best.pt) first |
| `lab_d_errors.csv` is empty | genuinely no errors at this threshold (or threshold too high) | report "0 errors" honestly; optionally analyze hard samples — never fabricate |
| Official P/R very different from fixed-threshold P/R | the two calibers are different by design | re-read the dual-caliber note above; cite them separately |
| PR/confusion plots missing | plots directory varies by version | open the printed `val_metrics.save_dir` path |

In [ ]:
from pathlib import Path
import os
import sys
import json

candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace/YOLO26_ROCm_Day1_Labs')]
package = next((p for p in candidates if (p / 'course/lab_utils.py').exists()), None)
if package is None:
    raise FileNotFoundError('请先将 ZIP 完整解压，并从项目目录或 notebooks/ 目录启动 Jupyter。')
sys.path.insert(0, str(package))
from course.lab_utils import (root, ensure_dirs, load_config, save_config, write_json,
                              write_csv, system_info, require_rocm, sha256)

home = ensure_dirs()
cfg = load_config()
print('Project:', package, '\nPersistent output:', home)

In [ ]:
from course.lab_utils import (get_metrics_summary, inspect_dataset, match_predictions,
                              xywhn_to_xyxy, read_labels, label_path_for_image)
from ultralytics import YOLO
from PIL import Image, ImageDraw
from IPython.display import display
import torch

device = require_rocm()
assert cfg['baseline_weights'] and Path(cfg['baseline_weights']).is_file(), \
    '先完成 Lab C，或经过核验设置 best.pt'
assert Path(cfg['dataset_yaml']).is_file()
model = YOLO(cfg['baseline_weights'])
print('Weight SHA256:', sha256(Path(cfg['baseline_weights'])))

In [ ]:
# 正式官方 mAP 计算：不要把 conf=0.25 当作计算 mAP 的唯一阈值
val_metrics = model.val(data=cfg['dataset_yaml'], split='val', device=device,
                        imgsz=640, batch=8, workers=0, plots=True, verbose=False)
score = get_metrics_summary(val_metrics)
score.update({'split': 'val', 'dataset_yaml': cfg['dataset_yaml'],
              'weight_sha256': sha256(Path(cfg['baseline_weights']))})
write_json(home / 'reports/lab_d_official_metrics.json', score)
print('Official validation metrics:', score)
print('mAP: IoU=0.50..0.95 average, NOT confidence 50%-95%')

In [ ]:
# 展示 Ultralytics 官方可视化（如果已生成）
from PIL import Image

out = Path(val_metrics.save_dir)
for title in ('BoxPR_curve.png', 'PR_curve.png', 'confusion_matrix.png',
              'confusion_matrix_normalized.png'):
    p = out / title
    if p.exists():
        print(title)
        display(Image.open(p).resize((950, 600)))

In [ ]:
# 手工逐图片匹配：一个 GT 最多配一个同类、IoU>=0.50 的预测，其他为 FP/FN
qa = inspect_dataset(Path(cfg['dataset_yaml']))
val_images = qa['sample_paths']['val']
rows = []
failures = []
totals = {'TP': 0, 'FP': 0, 'FN': 0}
for img in val_images:
    pred = model.predict(source=str(img), device=device, imgsz=640,
                         conf=0.25, iou=0.70, verbose=False)[0]
    gt = label_path_for_image(img)
    match = match_predictions(pred, gt, iou_threshold=0.50)
    for k in totals:
        totals[k] += match[k]
    for issue in match['errors']:
        rows.append({'image': img.name, 'image_path': str(img), 'error': issue['error'],
                     'class_id': issue['class_id'], 'score': issue['score'],
                     'max_iou_gt': issue['max_iou_gt']})
    if match['errors']:
        failures.append((img, pred, gt, match))
precision = totals['TP'] / max(1, totals['TP'] + totals['FP'])
recall = totals['TP'] / max(1, totals['TP'] + totals['FN'])
print('Fixed threshold teaching-level counts:', totals,
      'P:', round(precision, 3), 'R:', round(recall, 3))
print('Important: these are NOT the official integrated PR or mAP values.')
write_csv(home / 'reports/lab_d_errors.csv', rows,
          ['image', 'image_path', 'error', 'class_id', 'score', 'max_iou_gt'])
write_json(home / 'reports/lab_d_fixed_threshold.json',
           {'conf': 0.25, 'matching_iou': 0.50,
            'nms_iou': 0.70, 'counts': totals,
            'precision_fixed': precision, 'recall_fixed': recall,
            'note': 'Teaching matching only; not a COCO AP implementation.'})

In [ ]:
# 保存真实失败图（左真值，右预测）；如果几乎无错误，忠实报告而不是捏造 5 个。
from PIL import Image, ImageDraw

error_dir = home / 'assets/lab_d_errors'
error_dir.mkdir(parents=True, exist_ok=True)
names = list(qa['names'].values()) if isinstance(qa['names'], dict) else list(qa['names'])
for idx, (img, pred, lab, match) in enumerate(failures[:5], 1):
    gt = Image.open(img).convert('RGB')
    W, H = gt.size
    d = ImageDraw.Draw(gt)
    for box in read_labels(lab):
        coords = xywhn_to_xyxy(box, W, H)
        d.rectangle(coords, outline=(40, 210, 120), width=max(2, W // 160))
        d.text((coords[0], max(0, coords[1] - 15)), f'GT {names[box[0]]}', fill=(0, 170, 85))
    predicted = Image.fromarray(pred.plot()[..., ::-1]).convert('RGB')
    board = Image.new('RGB', (W * 2, H), (245, 248, 250))
    board.paste(gt, (0, 0))
    board.paste(predicted, (W, 0))
    p = error_dir / f'{idx:02d}_{img.stem}.jpg'
    board.save(p, quality=90)
    print(p, 'errors:', len(match['errors']))
    display(board.resize((960, round(960 * H / (2 * W)))))
if len(failures) < 5:
    print(f'只有 {len(failures)} 张有失败案例。不能编造；可拓展验证集或评估更多困难样本。')

### 学生工作：手动归因至少 5 个实际错误（如果有）
打开 `reports/lab_d_errors.csv` 和并排对照图，添加 **现象、证据、可能根因、下次实验验证方法**。把推测与已确认根因区分开。

下面提供一个可保存的归因练习格：从 `lab_d_errors.csv` 里挑**一个**你最好奇的错误，把四项分析写进 `MY_ANALYSIS` 再运行。练习格**默认可直接运行**（会先打印你选中错误行的真实证据）；写错不扣分——`filled` 字段会如实记录你是否完成了归因。

---

### Student Work: manually attribute at least 5 real errors (if any)
Open `reports/lab_d_errors.csv` and the side-by-side boards, then add **symptom, evidence, suspected root cause, and how the next experiment would verify it**. Keep hypotheses clearly separate from confirmed causes.

The practice cell below makes this concrete: pick **one** error you are most curious about from `lab_d_errors.csv`, fill the four fields of `MY_ANALYSIS`, then run it. The cell **runs by default** (it prints the real evidence of your chosen error row first); wrong hypotheses cost nothing — the `filled` flags honestly record whether you completed the attribution.

In [ ]:
# 学生练习：独立归因一个真实错误（默认可直接运行——先看证据，再写下你的归因）
import csv

with (home / 'reports/lab_d_errors.csv').open(encoding='utf-8-sig') as f:
    error_rows = list(csv.DictReader(f))
MY_ERROR_INDEX = 0  # 改成你想分析的那一行（0 到 len(error_rows)-1）
MY_ANALYSIS = {
    'symptom': 'TODO: 现象——这张图上到底发生了什么？',
    'evidence': 'TODO: 证据——引用该行的 error / score / max_iou_gt 数字与图上所见',
    'suspected_cause': 'TODO: 你的假设（遮挡/小目标/类别混淆/标注问题/阈值…）',
    'verify_next_step': 'TODO: 下一次实验如何验证这个假设？',
}
# 无人值守复验通道（学生不需要）：YOLO_PRACTICE_FILL 指向 JSON 可注入已填写样例
if os.environ.get('YOLO_PRACTICE_FILL'):
    _p = Path(os.environ['YOLO_PRACTICE_FILL'])
    if _p.is_file():
        for _k, _v in json.loads(_p.read_text(encoding='utf-8')).get('lab_d_error_analysis', {}).items():
            MY_ANALYSIS[_k] = _v
filled = {key: not str(value).startswith('TODO') for key, value in MY_ANALYSIS.items()}
assert error_rows, '没有错误行——恭喜，但请如实报告并挑一张困难样本做分析'
assert 0 <= MY_ERROR_INDEX < len(error_rows), f'索引需在 0 到 {len(error_rows) - 1} 之间'
chosen = error_rows[MY_ERROR_INDEX]
print('你选择的错误:', chosen['image'], chosen['error'], '| score:', chosen['score'],
      '| max_iou_gt:', chosen['max_iou_gt'])
if not all(filled.values()):
    print('提示：四字段还没填（TODO 未替换）。上面这行就是真实证据——打开 assets/lab_d_errors/ 里对应的'
          '失败对照图，把现象/证据/假设/验证写进 MY_ANALYSIS，再运行一次保存你的归因。')
write_json(home / 'reports/lab_d_my_error_analysis.json',
           {'chosen': chosen, 'analysis': MY_ANALYSIS, 'filled': filled})

### 思考题 / 面试必答
1. 为什么官方 mAP50 和单一 conf=0.25 下的 Precision 不是同一个东西？
2. IoU=0.5 的定位标准会漏掉什么信息？
3. `mAP50-95` 的 50–95 指置信度吗？
4. 为什么不能用测试集调阈值后再报告该测试集成绩？

**交付**：`lab_d_official_metrics.json`、`lab_d_fixed_threshold.json`、`lab_d_errors.csv`、`lab_d_my_error_analysis.json`、至少现有真实失败图和分析报告。

---

### Think / Interview Questions
1. Why is the official mAP50 not the same thing as Precision at a single conf=0.25?
2. What information does an IoU=0.5 localization criterion throw away?
3. Does the 50–95 in `mAP50-95` refer to confidence?
4. Why can't you tune thresholds on a test set and then report scores on that same set?

**Deliverables**: `lab_d_official_metrics.json`, `lab_d_fixed_threshold.json`, `lab_d_errors.csv`, `lab_d_my_error_analysis.json`, the real failure images so far and an analysis report.